# Maize Yield Prediction for Smallholder Farmers

**Mission:** help smallholder farmers, especially young farmers, grow more food with less guesswork.

**Use case:** predict maize yield (kg per hectare) from farm conditions and practices.

**Who uses it and what for:** farmers and agricultural extension officers use the predicted yield to plan which practices (fertiliser, planting date, seed type) to focus on before the season.

**Dataset:** One Acre Fund maize survey of smallholder fields in Sub-Saharan Africa, 2016-2022 (Zenodo: https://zenodo.org/doi/10.5281/zenodo.11115815), licence CC BY 4.0.

In [4]:
import pandas as pd

df = pd.read_csv("/One_Acre_Fund_MEL_maize_survey_data_2016-2022.csv", low_memory=False)
print(df.shape)
df.head()

(14773, 53)


,year,country,iso,season,cz,lon,lat,strat,avg_season_gdd,avg_season_ai,...,manure,fert_in_hole,lime_kg_ha,weeding,pesticide,disease,pest,striga,water_excess,yield_kg_ha
0,2016,kenya,KEN,first season,7701,34.067,0.227,01_KEN7701S1,2509.099854,0.998492,...,NaN,True,0.0,NaN,NaN,False,False,NaN,False,1462.951369
1,2017,kenya,KEN,first season,7701,34.100,0.238,01_KEN7701S1,2552.900146,1.038017,...,NaN,True,0.0,False,True,False,True,NaN,False,2817.771269
2,2019,kenya,KEN,first season,7701,34.082,0.247,01_KEN7701S1,2580.099854,1.008526,...,NaN,True,0.0,False,False,False,True,NaN,False,0.000000
3,2020,kenya,KEN,first season,7701,34.098,0.232,01_KEN7701S1,2562.800049,1.018445,...,True,False,0.0,True,False,False,True,True,False,4788.722787
4,2020,kenya,KEN,first season,7701,34.094,0.272,01_KEN7701S1,2612.800049,1.023641,...,NaN,False,0.0,False,NaN,False,False,True,False,2193.982268


### Missing values

Several columns have many blank values. The biggest gaps are in `manure` (13,887 rows blank), `striga` (12,249), `hyb_type` (12,191) and the other seed-trait columns (7,000 or more), because many farms did not record these details. Soil columns (such as `soil_pH` and `soil_clay`) are blank in 4,441 rows, and `row_spacing` in 6,334 rows.

Decision: I will drop columns that are blank in most rows, because there is too little real information to fill them in. For columns that are only partly blank but likely matter for yield (such as soil and planting details), I will fill the gaps in the next step instead of deleting thousands of farms.

In [5]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

,0
manure,13887
striga,12249
hyb_type,12191
hyb_yor,7903
comp_kg_ha,7726
hyb_tol_rust,7191
hyb_tol_msv,7191
hyb_tol_gls,7191
hyb_tol_ear_rot,7191
hyb_tol_mln,7191


### Maize yield (the target)

The average yield is about 2,969 kg per hectare and the middle value (median) is about 2,714. Yields vary a lot, with a standard deviation of about 1,686, and the highest is about 11,139. About 1.8% of fields have a yield of 0, which probably means a failed harvest.

Decision: I will keep the zero-yield fields, because crop failure is a real outcome that farmers face and the model should learn from it.

In [6]:
df["yield_kg_ha"].describe()

,yield_kg_ha
count,14773.000000
mean,2969.063101
std,1686.066889
min,0.000000
25%,1743.990232
50%,2713.866882
75%,3945.223578
max,11139.130825


### Feature selection

I kept 8 inputs: country, elevation, plant density (pl_m2), hybrid seed (yes/no), nitrogen (N_kg_ha), phosphorus (P_kg_ha), season rainfall and average season temperature. A farmer or extension officer can supply these, they have few or no blanks, and in a quick random forest test they were among the strongest predictors of yield.

Columns dropped and why:
- iso, cz, strat, lon, lat: codes or coordinates that duplicate country and elevation.
- plant_date, harvest_date: text dates, hard to use directly.
- manure, striga, compost, comp_kg_ha, row_spacing, hyb_* trait columns: blank in roughly 40% to 94% of rows.
- disease, pest, water_excess: only known after the season, so they cannot help plan ahead.
- Soil columns and the monthly rainfall columns: many blanks, or very similar to the rainfall and temperature columns I kept.

In [12]:
features = ["country", "elev", "pl_m2", "hybrid", "N_kg_ha", "P_kg_ha", "season_prec", "avg_season_tavg"]
target = "yield_kg_ha"

data = df[features + [target]].copy()
print(data.isna().sum())

data["hybrid"] = data["hybrid"].map({True: 1, False: 0, "True": 1, "False": 0})
data["hybrid"] = data["hybrid"].fillna(data["hybrid"].mode()[0])
data["pl_m2"] = data["pl_m2"].fillna(data["pl_m2"].median())
data["avg_season_tavg"] = data["avg_season_tavg"].fillna(data["avg_season_tavg"].median())
print("blanks left:", data.isna().sum().sum())

data = pd.get_dummies(data, columns=["country"], drop_first=True, dtype=int)
print(data.shape)
data.head()

country               0
elev                  0
pl_m2              4607
hybrid             1230
N_kg_ha               0
P_kg_ha               0
season_prec           0
avg_season_tavg      41
yield_kg_ha           0
dtype: int64
blanks left: 0
(14773, 14)


,elev,pl_m2,hybrid,N_kg_ha,P_kg_ha,season_prec,avg_season_tavg,yield_kg_ha,country_kenya,country_nigeria,country_rwanda,country_tanzania,country_uganda,country_zambia
0,1280,3.402686,1.0,51.892050,24.71050,677.039708,21.275661,1462.951369,1,0,0,0,0,0
1,1288,2.736283,1.0,51.892050,24.71050,760.532583,21.507408,2817.771269,1,0,0,0,0,0
2,1243,4.541739,1.0,27.873444,13.83788,724.153837,21.651323,0.000000,1,0,0,0,0,0
3,1271,3.986111,1.0,44.478900,49.42100,998.342599,21.559790,4788.722787,1,0,0,0,0,0
4,1226,2.277778,1.0,51.892050,24.71050,998.342599,21.824339,2193.982268,1,0,0,0,0,0


Missing values: hybrid was blank in 1,230 rows, so I filled it with the most common value (hybrid seed). pl_m2 was blank in about 4,600 rows and avg_season_tavg in 41 rows, so I filled both with the median. Burundi and Uganda have no plant density recorded at all, so those rows use the overall median, which makes the plant density less reliable there. Country was converted into 0/1 columns so the models can use it.